# Download Data

In [21]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize


In [2]:
data = pd.read_csv("fra_perfumes.csv")

In [3]:
data.head()

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
0,9am Afnanfor women,for women,3.73,174,"['citrus', 'musky', 'woody', 'aromatic', 'warm...",[],9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
1,9am Dive Afnanfor women and men,for women and men,4.29,842,"['fruity', 'woody', 'green', 'warm spicy', 'ar...",[],9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...
2,9am pour Femme Afnanfor women,for women,4.00,68,"['fruity', 'musky', 'amber', 'citrus', 'powder...",[],9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...
3,9pm Afnanfor men,for men,4.50,"6,865","['vanilla', 'amber', 'warm spicy', 'cinnamon',...",[],9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...
4,9pm pour Femme Afnanfor women,for women,3.49,63,"['woody', 'aromatic', 'rose', 'fruity', 'powde...",[],9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...


# Pre-Cleaning

## Text Cleaning

In [4]:
def bayesian_avg(R, v, C, m):
    """ R = perfume’s average rating
    v = number of votes (rating count)
    C = mean rating across all perfumes
    m = treshold, minimum vote count"""
    return (v/(v+m))*R + (m/(v+m))*C

def clean_text(text):
    return text.replace("[", "").replace("]", "").replace("'", "")

def clean_accords(a):
    if isinstance(a, str):
        return a.replace(",", " ").replace("/", " ")
    return

def clean_data(df):
    data = df.copy()
    
    ## Feature engineering

    data["Name"] = [b.replace(a , "").strip() for a, b in zip(data["Gender"].astype(str), data["Name"].astype(str))]
    data['Brand'] = data['Name'].apply(lambda x: x.split(' ')[-1])
    data["Name"] = [b.replace(a , "").strip() for a, b in zip(data["Brand"].astype(str), data["Name"].astype(str))]

    ## Handle missing values

    data.dropna(subset=["Description"], inplace=True)
    data.fillna("0", inplace=True)
    data[["Rating Count"]] = data[["Rating Count"]].replace(",", "", regex=True)
    data["Description"] = data["Description"].fillna("")
    data["Main Accords"] = data["Main Accords"].fillna("")
    
    ## Convert data types

    data[["Rating Count"]] = data[["Rating Count"]].astype(int)
    data[["Rating Value"]] = data[["Rating Value"]].astype(float)

    ## Standardize Gender

    gender_map = {"for women": "women", "for men": "men", "for women and men": "unisex"}
    data["Gender"] = data["Gender"].replace(gender_map)

    ## Clean text columns

    for col in ["Main Accords", "Perfumers"]:
        data[col] = data[col].apply(clean_text)

    data["Main Accords"] = data["Main Accords"].apply(clean_accords)

    ## Compute weighted rating

    C = data["Rating Value"].mean()
    m = data["Rating Count"].quantile(0.75)
    data["Weighted Rating"] = data.apply(lambda x: bayesian_avg(x["Rating Value"], x["Rating Count"], C, m), axis=1)

    return  data[['Name', 'Gender', 'Main Accords', 'Description', 'url', 'Weighted Rating', 'Brand']]

**bayesian_avg**  fuction solves reliability problem.

Example:

    * Perfume A: ⭐ 5.0 (2 ratings)

    * Perfume B: ⭐ 4.6 (500 ratings)

Perfume A looks better but it is not trustworth.  Here, this function penalizes the item with low rating. I use Bayesian averaging to account for rating reliability, preventing items with few ratings from being unfairly ranked too high.


**clean_data** is preprocessing funciton. It cleans data, prepares text for embeddings.  

I clean and decompose raw product data into semantically independent features, enabling controlled embedding fusion and reliable recommendation.




In [5]:
for col in ["Main Accords", "Perfumers"]:
    data[col] = data[col].apply(clean_text)

data["Description"] = data["Description"].fillna("")
data["Main Accords"] = data["Main Accords"].fillna("")

In [6]:
df_clean = clean_data(data)

/var/folders/nw/gghxs9bd0xx7_4l1shb5xtmc0000gn/T/ipykernel_2833/1130341299.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data.fillna("0", inplace=True)


I normalize structured text fields and replace missing values with empty strings to ensure stable and consistent embedding generation.

## Genre Map

In [7]:
gender_map = {"for women": "women", "for men": "men", "for women and men": "unisex"}
df_clean["Gender"] = df_clean["Gender"].replace(gender_map)

In [8]:
df_clean.head()

,Name,Gender,Main Accords,Description,url,Weighted Rating,Brand
0,9am,women,citrus musky woody aromatic warm spicy la...,9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...,3.692447,Afnan
1,9am Dive,unisex,fruity woody green warm spicy aromatic ci...,9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...,4.220751,Afnan
2,9am pour Femme,women,fruity musky amber citrus powdery sweet ...,9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...,3.779070,Afnan
3,9pm,men,vanilla amber warm spicy cinnamon sweet f...,9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...,4.487696,Afnan
4,9pm pour Femme,women,woody aromatic rose fruity powdery violet...,9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...,3.572643,Afnan


In [9]:
df_clean.to_pickle("data.pkl")

# Embedding

In [10]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

**Why I use SentenceTransformer instead of raw Bert ?**

Bert runs token level, need training, slow and not optimal for similarity.

SentenceTransformer runs sentence level, already trained, fast and trained for similarity.

**Why I use this model ?**

It is fast and strong.

## Genre Vector

In [11]:
GENDER_VECTORS = {
    "male":   np.array([1.0, 0.0]),
    "female": np.array([0.0, 1.0]),
    "unisex": np.array([1/np.sqrt(2), 1/np.sqrt(2)]),
}

In [12]:
def gender_to_vec(g):
    g = str(g).lower()
    return GENDER_VECTORS.get(g, np.array([0, 0, 1], dtype=float))

def expand_gender_vec(gv, dim=384):
    # Convert 3-dim gender vector → same dimensionality as BERT embedding
    # 384 is the dimensionality of the "all-MiniLM-L6-v2" model's output embeddings
    return np.pad(gv, (0, dim - len(gv)), 'constant')


In [13]:
gender_embs = np.vstack([expand_gender_vec(gender_to_vec(g)) for g in df_clean["Gender"]])

Here our gender data embedded into model space.*

## Text Embeddings

In [14]:
def embed_text(df, col, model):
    texts = df[col].fillna("").tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    return embeddings

In [15]:
desc_embs = embed_text(df_clean, "Description", model)
accord_embs = embed_text(df_clean, "Main Accords", model)

Batches: 100%|██████████| 2191/2191 [00:43<00:00, 50.34it/s]


## Brand Embedding

In [ ]:
def create_brand_matrix(df):
    data = df.copy()

    brands = sorted(data["Brand"].unique())
    brand_to_idx = {b: i for i, b in enumerate(brands)}

    brand_matrix = np.zeros((len(data), len(brands)), dtype=np.float32)

    for i, brand in enumerate(data["Brand"]):
        brand_matrix[i, brand_to_idx[brand]] = 1
    
    return brand_matrix


def generate_brand_embedings(df, shape = 384):
    brand_matrix = create_brand_matrix(df)

    pca = PCA(n_components=shape)
    brand_embs = pca.fit_transform(brand_matrix)
    return brand_embs

In [17]:
brand_embs = generate_brand_embedings(df_clean, shape=desc_embs.shape[1])

In [19]:
test_matrix = create_brand_matrix(df_clean)
test_matrix.shape

(70103, 1462)

Here **create_brand_matrix** function creates brand embeddings, but there is dimension problem. In order to fix it, I use pca.

# Saving Embeddings

In [23]:
desc_embs = normalize(desc_embs)        # MiniLM output
brand_embs = normalize(brand_embs)      # PCA brand embeddings
gender_embs = normalize(gender_embs)
accord_embs = normalize(accord_embs)

In [24]:
np.save("desc_embs.npy", desc_embs)
np.save("accord_embs.npy", accord_embs)
np.save("gender_embss.npy", gender_embs)
np.save("brand_embs.npy", brand_embs)